In [1]:
# ================================================================
# NOTEBOOK : nb_gold_daily_sales
# Reads    : silver_lakehouse → silver_sales, silver_store, silver_weather
# Writes   : gold_lakehouse  → gold_daily_sales
# Logic    : Daily revenue per store, enriched with weather
# ================================================================


StatementMeta(, 2dc556e6-8c5c-4c44-8fd0-4d25742e17a6, 3, Finished, Available, Finished, False)

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col,round 


# ── READ Silver tables ────────────────────────────────────────
sales = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_sales")


weather = (spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_weather")    # selecting specific columns
.select(col("forecast_date").alias("WeatherDate"),
                        col("city").alias("WeatherCity"),
                        "weather_main", "temp_c", "rain_3h_mm", "is_rainy")
)


store = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_store")\
.select("StoreID", "StoreName", "City", "Region", "StoreType")



print(f"[READ]: sales = {sales.count()} store = {store.count()} weather = {weather.count()}")


# ── STEP 1: Aggregate sales to daily grain per store ────────────
# Exclude returns from revenue

daily_agg = (
    sales
    .groupBy("TransactionDate", "StoreID", "Year", "Month", "DayOfWeek", "IsWeekend")
    .agg(F.sum(F.when(col("IsReturn") == False, col("TotalAmount")).otherwise(0)).alias("GrossRevenue"),
    F.sum("DiscountAmount").alias("TotalDiscount"),
    F.sum("NetRevenue").alias("NetRevenue"),
    F.count(F.when(col("IsReturn") == False,1).otherwise(0)).alias("TransactionCount"),     # TransactionCount
    F.sum(F.when(col("IsReturn") == True,1).otherwise(0)).alias("ReturnCount"),                     #ReturnCount
    F.countDistinct("ProductID").alias("UniqueProductsSold"),
    F.avg(F.when(col("IsReturn") == False, col("TotalAmount"))).alias("AvgBasketSize"),
    
    )
)

# ── STEP 2: Join Store dimension ─────────────────────────────────
gold_df = daily_agg.join(store, on = "storeID", how = "left")

# ── STEP 3: Join Weather (match store's city + transaction date) ─

gold_df =(
    gold_df.join(weather,
    (gold_df.TransactionDate == weather.WeatherDate) & (gold_df.City == weather.WeatherCity), 
    how = "left"
    )
.drop("WeatherDate", "WeatherCity")
)


# ── STEP 4: Derived business metrics ──────────────────────────────

gold_df = ( 
    gold_df
    .withColumn("DiscountRate", round(col("TotalDiscount") / F.when(col("GrossRevenue") == 0,1).otherwise(col("GrossRevenue")),4))
    .withColumn("ReturnRate", round(col("ReturnCount") / F.when(col("TransactionCount") == 0,1).otherwise(col("TransactionCount")),4))
    .withColumn("_gold_load_ts", F.current_timestamp())

)

# ── WRITE to gold_lakehouse ────────────────────────────────────

gold_df.write.format("delta").mode("overwrite")\
.option("overwriteSchema", "true")\
.saveAsTable("gold_daily_sales")


print(f"[DONE]gold_daily_sales:{gold_df.count()} rows")

display(gold_df.limit(5))







StatementMeta(, b27d18a0-55fb-46be-88c6-af8f89ed7a56, 3, Finished, Available, Finished, False)

[READ]: sales = 2000 store = 50 weather = 40
[DONE]gold_daily_sales:1552 rows


SynapseWidget(Synapse.DataFrame, eca8e8a3-7839-4215-9c29-149f6a8916a0)